In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Used_Bikes.csv')
df

,bike_name,price,city,kms_driven,owner,age,power,brand
0,TVS Star City Plus Dual Tone 110cc,35000.0,Ahmedabad,17654.0,First Owner,3.0,110.0,TVS
1,Royal Enfield Classic 350cc,119900.0,Delhi,11000.0,First Owner,4.0,350.0,Royal Enfield
2,Triumph Daytona 675R,600000.0,Delhi,110.0,First Owner,8.0,675.0,Triumph
3,TVS Apache RTR 180cc,65000.0,Bangalore,16329.0,First Owner,4.0,180.0,TVS
4,Yamaha FZ S V 2.0 150cc-Ltd. Edition,80000.0,Bangalore,10000.0,First Owner,3.0,150.0,Yamaha
...,...,...,...,...,...,...,...,...
32643,Hero Passion Pro 100cc,39000.0,Delhi,22000.0,First Owner,4.0,100.0,Hero
32644,TVS Apache RTR 180cc,30000.0,Karnal,6639.0,First Owner,9.0,180.0,TVS
32645,Bajaj Avenger Street 220,60000.0,Delhi,20373.0,First Owner,6.0,220.0,Bajaj
32646,Hero Super Splendor 125cc,15600.0,Jaipur,84186.0,First Owner,16.0,125.0,Hero


In [3]:
df.isna().sum()

bike_name     0
price         0
city          0
kms_driven    0
owner         0
age           0
power         0
brand         0
dtype: int64

In [4]:
x = df.drop(['price','brand'], axis='columns')
y = df['price']

In [5]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [6]:
def preprossing(x,y):
    encoder = OneHotEncoder(sparse_output=False)
    x_encoded = encoder.fit_transform(x[['bike_name','city','owner','power']])

    feature_names = encoder.get_feature_names_out()
    x_encoded_df = pd.DataFrame(x_encoded, columns=feature_names)

    join_x_encoded = x[['kms_driven','age']].join(x_encoded_df)

    scaler = StandardScaler()
    scaled = scaler.fit_transform(join_x_encoded)

    feature_names = join_x_encoded.columns
    x_encoded_df = pd.DataFrame(scaled, columns=feature_names)

    x_train, x_test, y_train, y_test = train_test_split(x_encoded_df, y, test_size=0.33, random_state=42)

    return x_train, x_test, y_train, y_test, encoder, scaler


In [7]:
x_train, x_test, y_train, y_test, encoder, scaler = preprossing(x,y)

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
import time
import warnings
from sklearn.metrics import r2_score

In [9]:
model_name = {
             'LinearRegression' : LinearRegression(),
             'Ridge' : Ridge(alpha=1.0),
             'Lasso' : Lasso(max_iter=1000),
             'DecisionTreeRegressor' : DecisionTreeRegressor(max_depth=5, random_state=42),
             'RandomForestRegressor' : RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
             }

In [10]:
def train_model(x):
    df = []
    for key, value in x.items():
        print('='*20)
        print(key)
        print('='*20)

        model_train = value

        #modeltime
        start_time = time.time()
        model = model_train.fit(x_train,y_train) 
        print(model)
        train_time = time.time() - start_time


        y_pred = model.predict(x_test)

        #train-test score
        train_score = model.score(x_train,y_train)
        test_score = model.score(x_test,y_test)

        #crosscal validation
        scores = cross_val_score(model, x_train, y_train, cv=5)
        cv_mean = scores.mean()
        cv_std = scores.std()

        #over fitting
        overfitting = train_score - test_score

        r2   = r2_score(y_test, y_pred)


        #print
        print(f'Train Score ={train_score}')
        print(f'Test Score={test_score}')
        print(f'cv_mean ={cv_mean}')
        print(f'cv_std ={cv_std}')
        print(f'Overfitting ={overfitting}')
        print(f'Training Time: {train_time}')
        print(f'r2 score: {r2}')


        #append 
        df.append({
        'model':key,
        'Train Score':train_score,
        'Test Score':test_score,
        'CV mean':cv_mean,
        'CV std':cv_std,
        'Overfitting':overfitting,
        'Train Time':train_time,
        'r2 score':r2,
        'train model' : model})

    df = pd.DataFrame(df)
    return df

In [11]:
results_df = train_model(model_name) 

LinearRegression
LinearRegression()
Train Score =0.9861099203721277
Test Score=0.8728596067131996
cv_mean =0.908827664900097
cv_std =0.021877076651224073
Overfitting =0.11325031365892813
Training Time: 1.915344476699829
r2 score: 0.8728596067131996
Ridge
Ridge()
Train Score =0.9861099135122514
Test Score=0.8728600100586961
cv_mean =0.908828630425013
cv_std =0.021881703320362378
Overfitting =0.11324990345355535
Training Time: 0.7509517669677734
r2 score: 0.8728600100586961
Lasso


C:\Users\ankit\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.209e+11, tolerance: 1.789e+10
  model = cd_fast.enet_coordinate_descent(


Lasso()


C:\Users\ankit\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.460e+11, tolerance: 1.414e+10
  model = cd_fast.enet_coordinate_descent(
C:\Users\ankit\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.342e+11, tolerance: 1.488e+10
  model = cd_fast.enet_coordinate_descent(
C:\Users\ankit\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.459e+11, toleranc

Train Score =0.9861098071700695
Test Score=0.8445017553595859
cv_mean =0.8758741934355584
cv_std =0.034803368738789885
Overfitting =0.14160805181048353
Training Time: 36.25549268722534
r2 score: 0.8445017553595859
DecisionTreeRegressor
DecisionTreeRegressor(max_depth=5, random_state=42)
Train Score =0.6659553878916061
Test Score=0.6324450313814687
cv_mean =0.607403225796636
cv_std =0.040967677207501794
Overfitting =0.03351035651013745
Training Time: 1.0410387516021729
r2 score: 0.6324450313814687
RandomForestRegressor
RandomForestRegressor(max_depth=5, random_state=42)
Train Score =0.7094105904244148
Test Score=0.6413267187738063
cv_mean =0.658543054618133
cv_std =0.03480588998826252
Overfitting =0.06808387165060847
Training Time: 27.87062406539917
r2 score: 0.6413267187738063


In [12]:
results_df

,model,Train Score,Test Score,CV mean,CV std,Overfitting,Train Time,r2 score,train model
0,LinearRegression,0.986110,0.872860,0.908828,0.021877,0.113250,1.915344,0.872860,LinearRegression()
1,Ridge,0.986110,0.872860,0.908829,0.021882,0.113250,0.750952,0.872860,Ridge()
2,Lasso,0.986110,0.844502,0.875874,0.034803,0.141608,36.255493,0.844502,Lasso()
3,DecisionTreeRegressor,0.665955,0.632445,0.607403,0.040968,0.033510,1.041039,0.632445,"DecisionTreeRegressor(max_depth=5, random_stat..."
4,RandomForestRegressor,0.709411,0.641327,0.658543,0.034806,0.068084,27.870624,0.641327,"(DecisionTreeRegressor(max_depth=5, max_featur..."


In [13]:
def best_model(df):
    #best model based on test score
    best_test_idx = df['Test Score'].idxmax()
    best_test_score = df.loc[best_test_idx,'model']
    print(f'the best model based on Train Score = {best_test_score}')
    print(f'the Score is = {df.loc[best_test_idx,'Test Score']:.5f}')

    #best model based on train score
    best_train_idx = df['Train Score'].idxmax()
    best_train_score = df.loc[best_train_idx, 'model']
    print(f'the best model based on Train Score = {best_train_score}')
    print(f'the Score is = {df.loc[best_train_idx,'Test Score']:.5f}')
    
    #best model based on cv mean
    best_cv_mean_idx = df['CV mean'].idxmax()
    best_cv_mean_score = df.loc[best_cv_mean_idx,'model']
    print(f'\nthe best model based on CV mean = {best_cv_mean_score}')
    print(f'the Score is = {df.loc[best_test_idx,'CV mean']:.5f}')
    
    #best model based on overfitting
    best_overfitting_idx = df['Overfitting'].abs().idxmin()
    best_overfitting_score = df.loc[best_overfitting_idx,'model']
    print(f'\nthe best model based on Overfitting = {best_overfitting_score}')
    print(f'the Score is = {df.loc[best_test_idx,'Overfitting']:.5f}')
    
    #best model based on training model
    best_train_time_idx = df['Train Time'].idxmin()
    best_train_time_score = df.loc[best_train_time_idx,'model']
    print(f'\nthe best model based on Train Time = {best_train_time_score}')
    print(f'nthe Score is = {df.loc[best_test_idx,'Train Time']:.5f}')
    
    #best overall model
    model_report = df.sort_values(by=['r2 score', 'Train Time', 'Overfitting'],
                                                 ascending=[False, True, True],
                                                 key=lambda x: x.abs() if x.name == 'Overfitting' else x)
    print(model_report)
    
    #Overall_best_model = df.loc[model_report,'model']
    best_overall_idx = model_report.index[0]
    print(best_overall_idx)
    best_overall_model = df.loc[best_overall_idx, 'model']
    print(f'\nthe Best Overall Model is = {df.loc[best_overall_idx, 'model']}')
    print(f'the r2 score is = {df.loc[best_overall_idx,'r2 score']}')
    print(f'the Train Time Score is = {df.loc[best_overall_idx,'Train Time']:.5}')
    print(f'the Overfitting Score is = {df.loc[best_overall_idx,'Overfitting']:.5f}')
    print(f'the CV mean Score is = {df.loc[best_overall_idx,'CV mean']:.5f}')
    print(f'the Test Score Score is = {df.loc[best_overall_idx,'Test Score']:.5f}')
    print(f'the Train Score is = {df.loc[best_overall_idx,'Train Score']:.5f}')

    perfect_model = model_name.get(best_overall_model)
    best_model = df.loc[best_overall_idx,'train model']
    return best_model

In [14]:
best_model_result = best_model(results_df)
best_model_result 

the best model based on Train Score = Ridge
the Score is = 0.87286
the best model based on Train Score = LinearRegression
the Score is = 0.87286

the best model based on CV mean = Ridge
the Score is = 0.90883

the best model based on Overfitting = DecisionTreeRegressor
the Score is = 0.11325

the best model based on Train Time = Ridge
nthe Score is = 0.75095
                   model  Train Score  Test Score   CV mean    CV std  \
1                  Ridge     0.986110    0.872860  0.908829  0.021882   
0       LinearRegression     0.986110    0.872860  0.908828  0.021877   
2                  Lasso     0.986110    0.844502  0.875874  0.034803   
4  RandomForestRegressor     0.709411    0.641327  0.658543  0.034806   
3  DecisionTreeRegressor     0.665955    0.632445  0.607403  0.040968   

   Overfitting  Train Time  r2 score  \
1     0.113250    0.750952  0.872860   
0     0.113250    1.915344  0.872860   
2     0.141608   36.255493  0.844502   
4     0.068084   27.870624  0.641327   


Ridge()

In [25]:
import sklearn

In [26]:
print(sklearn.__version__)

1.6.1


In [27]:
import pickle

In [28]:
with open('model.pkl', 'wb') as f:
    pickle.dump(best_model_result, f) 

In [29]:
with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

In [30]:
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)